# Demo New App Notebook
This notebook follows the structure of the original `Demo (Old App).ipynb` but loads and processes data directly from the JSON files in the Lineups project.

## Setup and Imports

In [1]:
import os
import glob
import json
import pandas as pd
from datetime import datetime

## Utility Functions

In [2]:
def load_json_to_df(filepath):
    """Load a JSON file and return a pandas DataFrame."""
    with open(filepath, 'r') as f:
        data = json.load(f)
    if isinstance(data, dict) and len(data) == 1 and isinstance(next(iter(data.values())), list):
        data_list = next(iter(data.values()))
    else:
        data_list = data
    return pd.json_normalize(data_list)

## Define Data Directory
Update this path to point to your JSON files directory

In [5]:
data_dir = r"C:\Users\pmalo\Desktop\Lineups v2"
json_files = glob.glob(os.path.join(data_dir, '*.json'))
print("Found JSON files:\n", json_files)

Found JSON files:
 ['C:\\Users\\pmalo\\Desktop\\Lineups v2\\allPlayByPlay_data.json', 'C:\\Users\\pmalo\\Desktop\\Lineups v2\\eligible_players.json', 'C:\\Users\\pmalo\\Desktop\\Lineups v2\\games.json', 'C:\\Users\\pmalo\\Desktop\\Lineups v2\\players.json']


## Load DataFrames

In [8]:
players_df = load_json_to_df(os.path.join(data_dir, 'players.json'))
eligible_players_df = load_json_to_df(os.path.join(data_dir, 'eligible_players.json'))
games_df = load_json_to_df(os.path.join(data_dir, 'games.json'))
play_by_play_df = load_json_to_df(os.path.join(data_dir, 'allPlayByPlay_data.json'))

print("Players:\n", players_df.head())
print("Eligible Players:\n", eligible_players_df.head())
print("Schedule:\n", games_df.head())
print("Play-by-Play:\n", play_by_play_df.head())

Players:
   pos playerID team             longName teamID
0   P   571656               Buck Farmer       
1   P   621121  HOU  Lance McCullers Jr.     11
2  LF   694362  CIN           Blake Dunn      7
3  RF   676551            Brewer Hicklen       
4   P   677960  MIA        Ryan Weathers     15
Eligible Players:
   pos playerID team          longName teamID
0  OF   694362  CIN        Blake Dunn      7
1  SS   669397  ATL        Nick Allen      2
2  OF   650559  ATL  Bryan De La Cruz      2
3  3B   805367  CHW    Chase Meidroth      6
4  2B   670032  LAA       Nicky Lopez     13
Schedule:
              gameID        gameType away gameTime  gameDate teamIDHome  \
0  20250415_OAK@CHW  REGULAR_SEASON  OAK    7:40p  20250415          6   
1  20250415_ARI@MIA  REGULAR_SEASON  ARI    6:40p  20250415         15   
2  20250415_ATL@TOR  REGULAR_SEASON  ATL    7:07p  20250415         29   
3   20250415_CHC@SD  REGULAR_SEASON  CHC    9:40p  20250415         23   
4  20250415_COL@LAD  REGULAR_SEA

## Core Functions

In [9]:
def get_eligible_players(date_str, eligible_df=eligible_players_df):
    """Return eligible players for a given date (YYYY-MM-DD)."""
    return eligible_df[eligible_df['date'] == date_str]


In [11]:
def get_schedule(date_str, games_df=games_df):
    """Return the scheduled matchups for a given date (YYYY-MM-DD)."""
    return games_df[games_df['date'] == date_str]


In [12]:
def create_lineup(team_id, eligible_df, lineup_size=9):
    """Randomly select a lineup of size `lineup_size` for `team_id`."""
    team_pool = eligible_df[eligible_df['teamID'] == team_id]
    return team_pool.sample(n=lineup_size, random_state=42)


In [13]:
def simulate_game(home_lineup, away_lineup, pbp_df):
    """Filter play-by-play data for players in both lineups."""
    players = set(home_lineup['playerID']) | set(away_lineup['playerID'])
    return pbp_df[pbp_df['playerID'].isin(players)]


In [14]:
def calculate_scores(sim_pbp):
    """Sum runs by team to get final scores."""
    return sim_pbp.groupby('teamID')['runs'].sum().to_dict()


## Demo Simulation

In [15]:
# Choose a date present in your JSON data
demo_date = '2024-07-15'

# Fetch eligible players and schedule
eligible_on_date = get_eligible_players(demo_date)
schedule_on_date = get_schedule(demo_date)

# Assume schedule has 'homeTeamID' and 'awayTeamID'
home_id = schedule_on_date.iloc[0]['homeTeamID']
away_id = schedule_on_date.iloc[0]['awayTeamID']

# Create lineups
home_lineup = create_lineup(home_id, eligible_on_date)
away_lineup = create_lineup(away_id, eligible_on_date)

# Simulate and score
game_pbp = simulate_game(home_lineup, away_lineup, play_by_play_df)
scores = calculate_scores(game_pbp)

print(f"Final scores on {demo_date}: {scores}")

KeyError: 'date'